In [4]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

I added edge-case checks to the initial filtering to handle invalid or duplicate tickers, missing or outdated prices, currency conversion issues, and missing financial information. I also added some extra checks to make sure the data is usable before passing it into the indicators.



In [5]:

import pandas as pd
import numpy as np
import yfinance as yf

# INITIAL PORTFOLIO FILTER
# 1. Must be a company stock listed in the US or Canada
# 2. Must trade in USD or CAD
# 3. Average volume must be at least 5000
# 4. Exclude months with fewer than 18 trading days

# Read the tickers
try:
    tickers = pd.read_csv("Tickers_Example.csv", header=None)
    tickers = tickers.dropna()
except Exception as e:
    raise ValueError("Could not read ticker file") from e

eligible = []
eligible_currency = []
checked = []  # NEW: track tickers already checked
stock_info = {}  # NEW: save stock information for later

for i in range(len(tickers)):
    # NEW: remove spaces and make tickers uppercase
    symbol = str(tickers.iloc[i, 0]).strip().upper()

    # NEW: skip empty or duplicate tickers
    if symbol == "" or symbol in checked:
        continue

    checked.append(symbol)
    valid = True

    try:
        stock = yf.Ticker(symbol)
        info = stock.info

        # Company stock?
        if info.get("quoteType") != "EQUITY":
            valid = False

        # Listed in US/Canada?
        if info.get("market") not in ["us_market", "ca_market"]:
            valid = False

        # Currency in USD/CAD?
        if info.get("currency") not in ["USD", "CAD"]:
            valid = False

        # Check average trading volume
        if valid:
            hist = stock.history(
                start="2025-10-01",
                end="2026-10-01"
            )

            # NEW: check if volume data is missing
            if hist.empty or "Volume" not in hist.columns:
                valid = False
            else:
                dates = pd.to_datetime(hist.index)
                volume = hist["Volume"].replace(
                    [np.inf, -np.inf], np.nan
                )

                # NEW: remove invalid volume values
                volume = volume.where(volume >= 0)

                days_per_month = volume.groupby(
                    [dates.year, dates.month]
                ).count()

                volume_per_month = volume.groupby(
                    [dates.year, dates.month]
                ).sum()

                total_days = 0
                total_volume = 0

                for m in range(len(days_per_month)):
                    if days_per_month.iloc[m] >= 18:
                        total_days += days_per_month.iloc[m]
                        total_volume += volume_per_month.iloc[m]

                if total_days == 0:
                    valid = False
                elif total_volume / total_days < 5000:
                    valid = False

    # NEW: handle failed ticker requests
    except Exception as e:
        print(symbol, "could not be checked:", e)
        valid = False

    if valid:
        eligible.append(symbol)
        eligible_currency.append(info.get("currency"))
        stock_info[symbol] = info


# Download historical prices
stock_data = {}

if len(eligible) > 0:
    try:
        all_stock = yf.download(
            eligible,
            start="2023-10-01",
            end="2026-11-28",
            group_by="ticker",
            auto_adjust=False,
            progress=False
        )

        for ticker in eligible:
            try:
                # NEW: handle single and multiple stock downloads
                if isinstance(all_stock.columns, pd.MultiIndex):
                    if ticker in all_stock.columns.get_level_values(0):
                        indiv_stock = all_stock[ticker].copy()
                    else:
                        indiv_stock = all_stock.xs(
                            ticker, axis=1, level=1
                        ).copy()
                else:
                    indiv_stock = all_stock.copy()

                required = ["High", "Low", "Close"]

                # NEW: check if any price columns are missing
                if not all(col in indiv_stock.columns for col in required):
                    print(ticker, "has missing price columns")
                    continue

                # NEW: remove rows with missing or invalid prices
                indiv_stock = indiv_stock.dropna(subset=required)

                for column in required:
                    indiv_stock = indiv_stock[
                        np.isfinite(indiv_stock[column]) &
                        (indiv_stock[column] > 0)
                    ]

                # NEW: check if enough historical data is available
                if len(indiv_stock) < 252:
                    print(ticker, "does not have enough price data")
                    continue

                # NEW: check if the stock has recent prices
                last_date = pd.Timestamp(indiv_stock.index[-1])
                last_date = last_date.tz_localize(None)

                check_date = min(
                    pd.Timestamp.today().normalize(),
                    pd.Timestamp("2026-11-27")
                )

                if (check_date - last_date).days > 7:
                    print(ticker, "has outdated price data")
                    continue

                indiv_stock.index = indiv_stock.index.strftime("%Y-%m-%d")
                stock_data[ticker] = indiv_stock

            # NEW: skip stocks with failed downloads
            except (KeyError, TypeError, ValueError) as e:
                print(ticker, "could not be downloaded:", e)

    # NEW: handle failed batch downloads
    except Exception as e:
        print("Historical download failed:", e)


# Convert USD prices to CAD
cadusd = pd.Series(dtype=float)

if len(stock_data) > 0:
    try:
        cadusd = yf.Ticker("CADUSD=X").history(
            start="2023-10-01",
            end="2026-11-28"
        )["Close"]

        cadusd.index = cadusd.index.strftime("%Y-%m-%d")

        # NEW: remove invalid exchange rates
        cadusd = cadusd.replace([np.inf, -np.inf, 0], np.nan)
        cadusd = cadusd.dropna()

    except Exception as e:
        print("Exchange rates could not be downloaded:", e)


# Convert the remaining stocks
for ticker in list(stock_data):
    table = stock_data[ticker]
    i = eligible.index(ticker)

    if eligible_currency[i] == "USD":
        # NEW: match exchange rates to stock dates
        dates = cadusd.index.union(table.index)
        rates = cadusd.reindex(dates).sort_index().ffill()
        table["CADUSD"] = rates.reindex(table.index)

        table["Close_CAD"] = table["Close"] / table["CADUSD"]
        table["High_CAD"] = table["High"] / table["CADUSD"]
        table["Low_CAD"] = table["Low"] / table["CADUSD"]

    else:
        table["Close_CAD"] = table["Close"]
        table["High_CAD"] = table["High"]
        table["Low_CAD"] = table["Low"]

    required = ["Close_CAD", "High_CAD", "Low_CAD"]

    # NEW: remove invalid CAD prices
    table[required] = table[required].replace(
        [np.inf, -np.inf], np.nan
    )

    table = table.dropna(subset=required)

    for column in required:
        table = table[table[column] > 0]

    # NEW: check that enough valid CAD prices remain
    if len(table) < 252:
        print(ticker, "does not have enough valid CAD prices")
        del stock_data[ticker]
        continue

    # NEW: check for recent CAD prices
    last_date = pd.Timestamp(table.index[-1])

    if (check_date - last_date).days > 7:
        print(ticker, "has outdated CAD prices")
        del stock_data[ticker]
        continue

    stock_data[ticker] = table


# NEW: update eligible lists after removing stocks
final_eligible = []
final_currency = []

for i in range(len(eligible)):
    if eligible[i] in stock_data:
        final_eligible.append(eligible[i])
        final_currency.append(eligible_currency[i])

eligible = final_eligible
eligible_currency = final_currency

print("Eligible tickers:", eligible)
print("Number of eligible stocks:", len(eligible))


Eligible tickers: ['AAPL', 'ABBV', 'ABT', 'ACN', 'AIG', 'AMZN', 'AXP', 'BA', 'BAC', 'BB.TO', 'BIIB', 'BLK', 'BMY', 'C', 'CAT', 'CL', 'KO', 'LLY', 'LMT', 'MO', 'MRK', 'PEP', 'PFE', 'RY.TO', 'PM', 'PYPL', 'QCOM', 'SHOP.TO', 'T.TO', 'TD.TO', 'TXN', 'UNH', 'UNP', 'UPS', 'USB']
Number of eligible stocks: 35


In [6]:

# Test the stock data after filtering

required = [
    "High", "Low", "Close",
    "High_CAD", "Low_CAD", "Close_CAD"
]

passed = 0
failed = 0

for ticker in stock_data:
    data = stock_data[ticker]

    if not all(col in data.columns for col in required):
        print(ticker, "FAILED: missing columns")
        failed += 1

    elif len(data) < 252:
        print(ticker, "FAILED: not enough price data")
        failed += 1

    elif data[required].isna().any().any():
        print(ticker, "FAILED: missing prices")
        failed += 1

    elif not np.isfinite(data[required].to_numpy()).all():
        print(ticker, "FAILED: invalid prices")
        failed += 1

    else:
        passed += 1

print("Passed:", passed)
print("Failed:", failed)


# Check if financial information is missing
print("\nMissing financial information:")

for ticker in stock_data:
    info = stock_info[ticker]
    missing = []

    if info.get("beta") is None:
        missing.append("Beta")

    if info.get("marketCap") is None:
        missing.append("Market Cap")

    if not info.get("sector"):
        missing.append("Sector")

    if missing:
        print(ticker, "missing:", ", ".join(missing))


Passed: 35
Failed: 0

Missing financial information:


When implementing the indicators into the main portfolio code, just make sure they're using the final eligible stocks and stock_data from the filtering, since some stocks might get removed during the checks. I also added indicator_issues to keep track of any missing financial information, so make sure to check that before running the indicators that need it. For correlation and portfolio risk, use cad_prices so everything is in CAD, and make sure the dates line up properly. The main thing is just to make sure all the indicators work with the filtered data and handle any missing values so we don't run into errors when building the portfolios.

In [7]:

# Check for missing information needed by indicators
indicator_issues = {}

for ticker in eligible:
    info = stock_info[ticker]
    missing = []

    if not info.get("sector"):
        missing.append("sector")

    for key in ["beta", "marketCap", "returnOnEquity", "trailingEps"]:
        value = info.get(key)

        if value is None:
            missing.append(key)
        elif not isinstance(value, (int, float)):
            missing.append(key)
        elif not np.isfinite(value):
            missing.append(key)
        elif key == "marketCap" and value <= 0:
            missing.append(key)

    if missing:
        indicator_issues[ticker] = missing
        print(ticker, "missing:", missing)


# Make a table with all CAD closing prices
cad_prices = pd.DataFrame()

for ticker in eligible:
    cad_prices[ticker] = stock_data[ticker]["Close_CAD"]

print("Stocks ready:", len(eligible))
print("Stocks with missing information:", len(indicator_issues))
print("CAD price table:", cad_prices.shape)


ABBV missing: ['returnOnEquity']
BA missing: ['returnOnEquity']
MO missing: ['returnOnEquity']
PM missing: ['returnOnEquity']
Stocks ready: 35
Stocks with missing information: 4
CAD price table: (789, 35)
